Definition of the absorption optical depth:
$$
\tau = log({I_0 \over I})
$$

Definition of the absorption Angstrom exponent:
$$
AAE = -{log(\tau_{450}/\tau_{600}) \over log(\lambda_{450}/\lambda_{600})}
$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
import ISSWlib as IS
import pandas as pd

In [2]:
%matplotlib inline

In [3]:
pwd

'/home/chemistry/LAI_in_snow/ISSW'

In [13]:
# Choose the spectrum folder and load in the list of spectra there
# spectrum_folder = 'Rainier_and_MtCook_comparison/'
# spectrum_folder = 'UPS_Rainier2025-07-03-01/'; filtertype = 'millipore'
# spectrum_folder = 'UCBoulder_Rainier/'; filtertype = 'millipore'
# spectrum_folder = 'UCBoulder_Baker/'; filtertype = 'millipore'
# spectrum_folder = 'UCBoulder_Adams/'; filtertype = 'millipore'
# spectrum_folder = 'UCBoulder_Misc/'; filtertype = 'millipore'
# spectrum_folder = 'UPS_2016Chile/'; filtertype = 'nucleopore'
# spectrum_folder = 'NuCStandards/'; filtertype = 'nucleopore'
# spectrum_folder = 'UPS_INKStandards/'; filtertype = 'nucleopore'
spectrum_folder = '2026July7/NucInc/'; filtertype = 'nucleopore'
# spectrum_folder = '2026July7/NucFul/'; filtertype = 'nucleopore'
# spectrum_folder = '2026July7/NucInc/'; filtertype = 'nucleopore'

# !ls -l '2026July7'
# !ls -l '2026July7/NucFul'
# !ls -l '2026July7/NucInc'

In [17]:
# Checking location of files
!ls '2026July7/NucInc/'

Blank.txt     NucA28mL.txt  NucC14mL.txt  NucC50mL.txt
NucA14mL.txt  NucB10mL.txt  NucC20mL.txt  NucC5mL.txt
NucA17mL.txt  NucB5mL.txt   NucC34mL.txt  spectrum_files_complete.txt
NucA24mL.txt  NucC10mL.txt  NucC40mL.txt  spectrum_files.txt


In [5]:
# Parameter file
parameter_filename = 'calibration parameters from UPS_INKStandards (July 13, 2026).csv'
# parameter_filename = 'calibration parameters (July 5, 2026).csv'
df = pd.read_csv(parameter_filename)
print(df)
LRF = df.at[0, 'LRF']
R1R2_1 = df.at[0,'R1R2_1']
R1R2_2 = df.at[0,'R1R2_2']
beta1_std = df.at[0,'beta1_std']
beta2_std = df.at[0,'beta2_std']
AAE_std = df.at[0,'AAE_std']
LAHM_factor = df.at[0,'LAHM_factor']

            Standard       LRF    R1R2_1    R1R2_2  beta1_std  beta2_std  \
0  UPS_INKStandards/  0.390909  0.534281  0.602341          5   3.837305   

   AAE_std  LAHM_factor  
0     0.92     0.463158  


In [6]:
spectrum_filelist = spectrum_folder+'spectrum_files.txt'
spectrum_list, number_of_loadings = IS.get_spectrum_list(spectrum_filelist)

FileNotFoundError: [Errno 2] No such file or directory: '2026July7/NucInk/spectrum_files.txt'

In [ ]:
# Load in the blank, and check indices
lambda_nm, I0_raw = IS.get_spectrum(spectrum_folder+'/Blank.txt')
I_450 = 100; print('lambda_450 = ',lambda_nm[I_450])
I_600 = 250; print('lambda_600 = ',lambda_nm[I_600])
I_1 = I_450
I_2 = I_600

In [ ]:
# Preallocate arrays and other constants
chi_observed = np.zeros((number_of_loadings,2))
bot = np.log(lambda_nm[I_1]/lambda_nm[I_2])

In [ ]:
for i in range(number_of_loadings):
    
    # Extract the spectrum for this item in the list
    spectrum_filename = spectrum_list[i]
    lambda_nm, I_raw = IS.get_spectrum(spectrum_folder+spectrum_filename)
    
    # Smooth and shift
    I, I0 = IS.smooth_and_shift(I_raw,I0_raw)
    
    # Get the observed chi-values
    chi = IS.get_chi_obs(I,I0,lambda_nm, title=spectrum_filename)
    print('chi1, chi2 = ', chi[I_1],chi[I_2])
    
    # Save chi values at lambda1 and lambda2
    chi_observed[i,0] = chi[I_1]
    chi_observed[i,1] = chi[I_2]
    
    # Get the tau values associated with these two points
    tau_1 = IS.invert_chi_theory(chi[I_1],R1R2_1)
    tau_2 = IS.invert_chi_theory(chi[I_2],R1R2_2)
    
    # Get the Angstrom exponent
    top = np.log(tau_1/tau_2)
    AAE = -top/bot
    
    # Record for plotting later
    if i==0:
        AAElist = AAE
        Ilist = I
        tau_1list = tau_1
        tau_2list = tau_2
    else:
        AAElist = np.append(AAElist,AAE)
        Ilist = np.vstack((Ilist,I))
        tau_1list = np.append(tau_1list,tau_1)
        tau_2list = np.append(tau_2list,tau_2)

In [ ]:
# Getting equivalent loadings
L_1list = tau_1list/(LRF*beta1_std/100)
L_2list = tau_2list/(LRF*beta2_std/100)

# Reporting
print('Loadings and AAE, nucleopore')
for i in range(number_of_loadings):
    print('For ',spectrum_list[i],', L1 =', L_1list[i],', L2 =', L_2list[i],', AAE =', AAElist[i])


# Adjusting for the filter type; if it's LAHM, the units become ug (instead of ug/cm^2)
if filtertype == 'millipore':
    print('')
    print('Applying the millipore filter correction ...')
    L_1list_LAHM = L_1list*LAHM_factor
    L_2list_LAHM = L_2list*LAHM_factor

    # Reporting
    print('Loadings and AAE, millipore')
    for i in range(number_of_loadings):
        print('For ',spectrum_list[i],', L1 =', L_1list_LAHM[i],', L2 =', L_2list_LAHM[i],', AAE =', AAElist[i])